# 04 — Gold Aggregations

Build analytics-ready datasets from the Silver layer and persist them to the Gold layer.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as _sum, date_trunc

spark = SparkSession.builder \
    .appName("lakehouse-gold") \
    .config(
        "spark.jars.packages",
        "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262"
    ) \
    .config("spark.hadoop.fs.s3a.endpoint", "http://lakehouse-minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "password123") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .getOrCreate()

In [ ]:
silver_path = "s3a://lakehouse/silver/online_retail/"
gold_country_path = "s3a://lakehouse/gold/country_revenue/"
gold_monthly_path = "s3a://lakehouse/gold/monthly_revenue/"

df_silver = spark.read.parquet(silver_path)
df_silver.show(5)

In [ ]:
df_country_revenue = (
    df_silver
    .groupBy("Country")
    .agg(_sum("Revenue").alias("TotalRevenue"))
    .orderBy(col("TotalRevenue").desc())
)

df_country_revenue.show(10, truncate=False)

In [ ]:
df_monthly_revenue = (
    df_silver
    .withColumn("RevenueMonth", date_trunc("month", col("InvoiceDate")))
    .groupBy("RevenueMonth")
    .agg(_sum("Revenue").alias("TotalRevenue"))
    .orderBy("RevenueMonth")
)

df_monthly_revenue.show(12, truncate=False)

In [ ]:
df_country_revenue.write.mode("overwrite").parquet(gold_country_path)
df_monthly_revenue.write.mode("overwrite").parquet(gold_monthly_path)

In [ ]:
spark.read.parquet(gold_country_path).show(10, truncate=False)
spark.read.parquet(gold_monthly_path).show(12, truncate=False)

## Result

The Gold layer now contains aggregated datasets suitable for BI, dashboards, or warehouse serving.